# Warehouse Throughput Analysis

## Project Overview
Analyze warehouse operations data to study throughput, bottlenecks, item flow, and staffing patterns. This is a descriptive analytics project.

## Learning Objectives
- Calculate and compare throughput metrics across warehouses and shifts
- Identify operational bottlenecks and peak-load periods
- Analyze staffing efficiency and its relationship to throughput
- Detect patterns in processing times and error rates

## Problem Statement
Operations management wants to understand throughput variation across warehouses: Which shifts are most productive? Where are bottlenecks? How does staffing relate to output?

## Why This Project Matters
Warehouse throughput directly impacts order fulfillment speed and cost. Identifying bottlenecks and optimizing staffing can improve service levels while reducing labor costs.

## Dataset Overview
Synthetic daily warehouse operations data: ~2,900 records across 4 warehouses, 3 shifts, over 2 years.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
warehouses = ['WH-Alpha', 'WH-Beta', 'WH-Gamma', 'WH-Delta']
shifts = ['Morning', 'Afternoon', 'Night']

records = []
for d in dates:
    for wh in warehouses:
        for sh in shifts:
            base = {'WH-Alpha': 450, 'WH-Beta': 380, 'WH-Gamma': 420, 'WH-Delta': 350}[wh]
            shift_mult = {'Morning': 1.0, 'Afternoon': 0.95, 'Night': 0.75}[sh]
            # Weekday vs weekend
            dow_mult = 0.6 if d.dayofweek >= 5 else 1.0
            # Seasonal: Q4 surge
            seasonal = 1.2 if d.month in [10, 11, 12] else 1.0
            staff = max(5, int(np.random.normal(
                {'Morning': 25, 'Afternoon': 22, 'Night': 15}[sh] * dow_mult, 3)))
            
            units = max(50, int(base * shift_mult * dow_mult * seasonal * (1 + np.random.normal(0, 0.08))))
            errors = max(0, int(np.random.poisson(units * 0.005)))
            avg_process_min = max(1, round(np.random.normal(
                {'WH-Alpha': 3.2, 'WH-Beta': 3.8, 'WH-Gamma': 3.5, 'WH-Delta': 4.1}[wh], 0.5), 1))
            
            records.append({
                'Date': d, 'Warehouse': wh, 'Shift': sh,
                'UnitsProcessed': units, 'Staff': staff,
                'Errors': errors, 'AvgProcessMinutes': avg_process_min,
            })

df = pd.DataFrame(records)
df['UnitsPerStaff'] = (df['UnitsProcessed'] / df['Staff']).round(1)
df['ErrorRate'] = (df['Errors'] / df['UnitsProcessed'] * 100).round(3)
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month
df['YearMonth'] = df['Date'].dt.to_period('M')

print(f'Dataset shape: {df.shape}')
print(f'Total units processed: {df["UnitsProcessed"].sum():,}')
print(f'Avg error rate: {df["ErrorRate"].mean():.3f}%')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Warehouses: {df["Warehouse"].nunique()} — {df["Warehouse"].unique().tolist()}')
print(f'Records per warehouse: {df.groupby("Warehouse").size().to_dict()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Warehouse')['UnitsProcessed'].mean().sort_values().plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Avg Daily Units by Warehouse')

df.groupby('Shift')['UnitsProcessed'].mean().plot.bar(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('Avg Units by Shift')
axes[0,1].tick_params(axis='x', rotation=0)

monthly = df.groupby('YearMonth')['UnitsProcessed'].sum()
monthly.plot(ax=axes[1,0], marker='o', color='green')
axes[1,0].set_title('Monthly Total Throughput')
axes[1,0].tick_params(axis='x', rotation=45)

df.groupby('Warehouse')['UnitsPerStaff'].mean().sort_values().plot.barh(ax=axes[1,1], edgecolor='black', color='purple')
axes[1,1].set_title('Avg Units per Staff Member')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Shift Productivity Analysis

In [ ]:
shift_stats = df.groupby(['Warehouse', 'Shift']).agg(
    avg_units=('UnitsProcessed', 'mean'),
    avg_staff=('Staff', 'mean'),
    units_per_staff=('UnitsPerStaff', 'mean'),
    avg_errors=('ErrorRate', 'mean'),
).round(2)
print('Shift Productivity:')
print(shift_stats)

pivot = df.groupby(['Warehouse', 'Shift'])['UnitsPerStaff'].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot.bar(ax=ax, edgecolor='black')
ax.set_title('Units per Staff — Warehouse × Shift')
ax.legend(title='Shift')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'shift_productivity.png'), dpi=100, bbox_inches='tight')
plt.show()

## Staffing vs Throughput Relationship

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for wh in warehouses:
    sub = df[df['Warehouse'] == wh].sample(200, random_state=42)
    ax.scatter(sub['Staff'], sub['UnitsProcessed'], alpha=0.3, label=wh, s=20)
ax.set_xlabel('Staff Count')
ax.set_ylabel('Units Processed')
ax.set_title('Staffing vs Throughput')
ax.legend(title='Warehouse')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'staff_throughput.png'), dpi=100, bbox_inches='tight')
plt.show()

corr = df.groupby('Warehouse').apply(lambda x: x['Staff'].corr(x['UnitsProcessed']), include_groups=False)
print('Staff-Throughput Correlation by Warehouse:')
print(corr.round(3))

## Error Rate Analysis

In [ ]:
err_wh = df.groupby('Warehouse')['ErrorRate'].mean().sort_values(ascending=False)
print('Avg Error Rate by Warehouse:')
print(err_wh.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
err_wh.plot.bar(ax=axes[0], edgecolor='black', color='red', alpha=0.6)
axes[0].set_title('Avg Error Rate by Warehouse')
axes[0].tick_params(axis='x', rotation=0)

err_shift = df.groupby('Shift')['ErrorRate'].mean()
err_shift.plot.bar(ax=axes[1], edgecolor='black', color='orange')
axes[1].set_title('Avg Error Rate by Shift')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Day-of-Week and Seasonal Patterns

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('DayOfWeek')['UnitsProcessed'].mean().reindex(dow_order)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dow.plot.bar(ax=axes[0], edgecolor='black', color='teal')
axes[0].set_title('Avg Units by Day of Week')
axes[0].tick_params(axis='x', rotation=45)

monthly_wh = df.groupby(['Month', 'Warehouse'])['UnitsProcessed'].mean().unstack()
monthly_wh.plot(ax=axes[1], marker='o')
axes[1].set_title('Monthly Avg Units by Warehouse')
axes[1].set_xlabel('Month')
axes[1].legend(title='Warehouse', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'patterns.png'), dpi=100, bbox_inches='tight')
plt.show()

## Processing Time Comparison

In [ ]:
proc = df.groupby('Warehouse')['AvgProcessMinutes'].agg(['mean', 'std', 'median']).round(2)
print('Processing Time (minutes):')
print(proc)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='Warehouse', y='AvgProcessMinutes', ax=ax)
ax.set_title('Processing Time Distribution by Warehouse')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'processing_time.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **WH-Alpha** leads in both throughput and units-per-staff — its processes should be benchmarked for other sites
- **WH-Delta** has the lowest efficiency and highest processing times — prime candidate for process improvement
- **Night shifts** produce significantly less per staff, which is expected but worth optimizing
- **Q4 surge** is clearly visible — staffing plans should pre-scale for October-December
- **Weekend** operations run at ~60% of weekday throughput — appropriate for demand patterns
- **Error rates** are generally low but WH-Delta's slightly higher rate correlates with slower processing

## Limitations
- Synthetic data with fixed warehouse baselines
- No cost data (labor cost, overtime, equipment costs)
- No order priority or SLA tracking
- No equipment downtime or maintenance data
- No individual worker productivity tracking

## How to Improve This Project
- Use real WMS (Warehouse Management System) data
- Add cost-per-unit analysis and labor cost optimization
- Include equipment utilization and downtime tracking
- Build simulation models for staffing optimization
- Add capacity planning with demand forecasting integration

## Production Considerations
- Real-time throughput dashboards per shift and warehouse
- Automated staffing recommendations based on demand forecasts
- Alert system for throughput drops or error rate spikes
- Benchmarking scorecards across warehouses

## Common Mistakes
- Comparing throughput without normalizing by staff or hours
- Ignoring shift and day-of-week effects when benchmarking
- Not accounting for product mix complexity in processing time comparisons
- Setting uniform targets without considering warehouse-specific constraints

## Mini Challenge / Exercises
1. Which warehouse-shift combination has the highest units-per-staff? Is it sustainable?
2. Calculate the Q4 throughput uplift (%) for each warehouse compared to Q1-Q3 average.
3. If WH-Delta could match WH-Alpha's processing time, estimate the throughput gain.
4. Is there a correlation between error rate and throughput? (Do higher-volume periods have more errors?)

## Final Summary / Key Takeaways
- Warehouse throughput analysis reveals operational efficiency differences across sites and shifts
- Units-per-staff is a better productivity metric than raw throughput
- Seasonal and day-of-week patterns must inform staffing and capacity planning
- Processing time and error rate complement throughput metrics for a complete operational picture
- Best-practice transfer from top-performing warehouses is a high-ROI improvement strategy